In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model

print("✅ TensorFlow version:", tf.__version__)
print("✅ Keras import successful!")



✅ TensorFlow version: 2.15.0
✅ Keras import successful!


In [2]:
import torch
from ultralytics import YOLO
from transformers import AutoConfig, AutoImageProcessor, SiglipForImageClassification
import cv2
import numpy as np

# -----------------------------
# Model Selector
# -----------------------------
class ModelSelector:
    def __init__(self, model_type):
        self.model_type = model_type

        if self.model_type == "sign":
            self._load_sign_model()
        elif self.model_type == "emotion":
            self._load_emotion_model()
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")

    # -----------------------------
    # Load Sign Language Model
    # -----------------------------
    def _load_sign_model(self):
        print("🔹 Loading Sign Language model...")
        config = AutoConfig.from_pretrained("sign_model")
        self.model = SiglipForImageClassification(config)

        state_dict = torch.load(
            r"C:\Users\anike\OneDrive\Desktop\ds project\HumanPatterns\sign_model\model_weights.pt",
            map_location="cpu"
        )
        self.model.load_state_dict(state_dict, strict=False)
        self.model.eval()

        self.processor = AutoImageProcessor.from_pretrained("sign_model", use_fast=False)
        print("✅ Sign model loaded successfully")

    # -----------------------------
    # Load Emotion YOLO Classification Model
    # -----------------------------
    def _load_emotion_model(self):
        print("🔹 Loading Emotion model (YOLO)...")
        self.model = YOLO(r"C:\Users\anike\OneDrive\Desktop\ds project\HumanPatterns\best.pt")
        print("✅ Emotion model loaded successfully (YOLO)")

    # -----------------------------
    # Predict
    # -----------------------------
    def predict(self, image):
        if self.model_type == "sign":
            # Prepare image for Siglip
            inputs = self.processor(images=image, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**inputs)
                preds = outputs.logits.argmax(-1).item()
            return preds

        elif self.model_type == "emotion":
            # Convert BGR → RGB
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # YOLO handles preprocessing internally
            results = self.model.predict(image_rgb, imgsz=640, verbose=False)

            # YOLO classification returns results[0].probs
            if results and results[0].probs is not None:
                pred_idx = results[0].probs.argmax().item()
                pred_name = results[0].names[pred_idx]
            else:
                pred_name = "Unknown"
            return pred_name

In [3]:
class RAGRouter:
    def __init__(self):
        self.sign_selector = ModelSelector("sign")
        #self.gender_selector = ModelSelector("gender")
        self.emotion_selector = ModelSelector("emotion")

    def run_webcam(self):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("⚠️ Cannot open webcam.")
            return

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Process frame (resize for inference)
            resized = cv2.resize(frame, (224, 224))

            # Get predictions
            sign_pred = self.sign_selector.predict(resized)
            emotion_pred = self.emotion_selector.predict(resized)
            #gender_pred = self.gender_selector.predict(resized)

            # Display predictions
            cv2.putText(frame, f"Sign: {sign_pred}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(frame, f"Emotion: {emotion_pred}", (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
            #cv2.putText(frame, f"Gender: {gender_pred}", (20, 120),
                       # cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            cv2.imshow("RAG Router", frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()


In [4]:
if __name__ == "__main__":
    router = RAGRouter()
    router.run_webcam()

🔹 Loading Sign Language model...
✅ Sign model loaded successfully
🔹 Loading Emotion model (YOLO)...
✅ Emotion model loaded successfully (YOLO)
